# Campus-Wide Sustainability Tracker

This notebook builds a simple ensemble forecasting pipeline on aggregated campus consumption data and presents KPI-focused dashboard views.

## Scope
- Aggregate building-level utility data to campus and zone levels
- Ensemble a basic regression model and exponential smoothing forecast
- Report sustainability KPIs including estimated carbon savings
- Provide drill-down visual views for campus operations

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
np.random.seed(42)

## 1) Build Demonstration Dataset
The dataset is generated to represent daily utility activity across multiple zones and buildings.

In [ ]:
dates = pd.date_range('2025-01-01', periods=365, freq='D')
zones = ['Academic', 'Residential', 'Admin', 'Recreation']
buildings = {
    'Academic': ['Block-A', 'Block-B', 'Library'],
    'Residential': ['Hostel-1', 'Hostel-2'],
    'Admin': ['Main-Office', 'Finance-Wing'],
    'Recreation': ['Sports-Complex', 'Auditorium']
}

rows = []
for current_date in dates:
    day_index = (current_date - dates.min()).days
    seasonality = 1 + 0.18 * np.sin(2 * np.pi * day_index / 30)
    weekly = 0.92 if current_date.weekday() >= 5 else 1.00

    for zone in zones:
        zone_factor = {'Academic': 1.25, 'Residential': 1.10, 'Admin': 0.85, 'Recreation': 1.00}[zone]
        for building in buildings[zone]:
            noise = np.random.normal(0, 25)
            base = 280 * zone_factor * seasonality * weekly
            energy_kwh = max(base + noise, 80)
            water_kl = max(0.09 * energy_kwh + np.random.normal(0, 3), 8)
            waste_kg = max(0.06 * energy_kwh + np.random.normal(0, 2), 5)
            renewable_kwh = max(0.20 * energy_kwh + np.random.normal(0, 7), 0)

            rows.append({
                'date': current_date,
                'zone': zone,
                'building': building,
                'energy_kwh': energy_kwh,
                'water_kl': water_kl,
                'waste_kg': waste_kg,
                'renewable_kwh': renewable_kwh
            })

building_daily = pd.DataFrame(rows)
building_daily.head()

## 2) Aggregate Campus Data and Prepare Features

In [ ]:
campus_daily = (
    building_daily
    .groupby('date', as_index=False)[['energy_kwh', 'water_kl', 'waste_kg', 'renewable_kwh']]
    .sum()
)

campus_daily['day_index'] = np.arange(len(campus_daily))
campus_daily['dow'] = campus_daily['date'].dt.dayofweek
campus_daily['is_weekend'] = (campus_daily['dow'] >= 5).astype(int)
campus_daily['month'] = campus_daily['date'].dt.month
campus_daily['sin_30'] = np.sin(2 * np.pi * campus_daily['day_index'] / 30)
campus_daily['cos_30'] = np.cos(2 * np.pi * campus_daily['day_index'] / 30)

feature_cols = ['day_index', 'sin_30', 'cos_30', 'is_weekend']
target_col = 'energy_kwh'

horizon = 30
train_df = campus_daily.iloc[:-horizon].copy()
val_df = campus_daily.iloc[-horizon:].copy()

train_df.tail(3), val_df.head(3)

## 3) Ensemble Forecasting: Regression + Smoothing
The ensemble weight is determined from validation MAE.

In [ ]:
def mae(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    return np.mean(np.abs(actual - predicted))

X_train = train_df[feature_cols].to_numpy()
y_train = train_df[target_col].to_numpy()
X_val = val_df[feature_cols].to_numpy()
y_val = val_df[target_col].to_numpy()

X_train_design = np.column_stack([np.ones(len(X_train)), X_train])
X_val_design = np.column_stack([np.ones(len(X_val)), X_val])

beta, *_ = np.linalg.lstsq(X_train_design, y_train, rcond=None)
reg_val_pred = X_val_design @ beta

alpha = 0.30
smooth_series = train_df[target_col].ewm(alpha=alpha, adjust=False).mean()
smooth_forecast_level = smooth_series.iloc[-1]
smooth_val_pred = np.repeat(smooth_forecast_level, len(val_df))

mae_reg = mae(y_val, reg_val_pred)
mae_smooth = mae(y_val, smooth_val_pred)

inv_errors = np.array([1 / mae_reg, 1 / mae_smooth])
weights = inv_errors / inv_errors.sum()
w_reg, w_smooth = weights
ensemble_val_pred = w_reg * reg_val_pred + w_smooth * smooth_val_pred

performance = pd.DataFrame({
    'model': ['Regression', 'Smoothing', 'Ensemble'],
    'MAE': [mae_reg, mae_smooth, mae(y_val, ensemble_val_pred)]
})
performance

In [ ]:
future_dates = pd.date_range(campus_daily['date'].max() + pd.Timedelta(days=1), periods=horizon, freq='D')
future_day_index = np.arange(campus_daily['day_index'].max() + 1, campus_daily['day_index'].max() + 1 + horizon)

future_df = pd.DataFrame({
    'date': future_dates,
    'day_index': future_day_index
})
future_df['is_weekend'] = (future_df['date'].dt.dayofweek >= 5).astype(int)
future_df['sin_30'] = np.sin(2 * np.pi * future_df['day_index'] / 30)
future_df['cos_30'] = np.cos(2 * np.pi * future_df['day_index'] / 30)

X_future = future_df[feature_cols].to_numpy()
X_future_design = np.column_stack([np.ones(len(X_future)), X_future])

future_reg_pred = X_future_design @ beta
future_smooth_pred = np.repeat(smooth_forecast_level, horizon)
future_ensemble_pred = w_reg * future_reg_pred + w_smooth * future_smooth_pred

forecast_df = future_df[['date']].copy()
forecast_df['regression_forecast_kwh'] = future_reg_pred
forecast_df['smoothing_forecast_kwh'] = future_smooth_pred
forecast_df['ensemble_forecast_kwh'] = future_ensemble_pred
forecast_df.head()

## 4) KPI Layer
KPIs summarize recent campus performance and forecasted impact.

In [ ]:
recent_30 = campus_daily.tail(30).copy()
forecast_30 = forecast_df.copy()

grid_emission_factor_baseline = 0.82
grid_emission_factor_current = 0.70

actual_recent_energy = recent_30['energy_kwh'].sum()
forecast_energy_next_30 = forecast_30['ensemble_forecast_kwh'].sum()
renewable_share_recent = (recent_30['renewable_kwh'].sum() / actual_recent_energy) * 100

carbon_baseline_tonnes = forecast_energy_next_30 * grid_emission_factor_baseline / 1000
carbon_projected_tonnes = forecast_energy_next_30 * grid_emission_factor_current / 1000
carbon_savings_tonnes = carbon_baseline_tonnes - carbon_projected_tonnes

kpi_table = pd.DataFrame({
    'KPI': [
        'Actual Energy (Last 30 Days) kWh',
        'Forecast Energy (Next 30 Days) kWh',
        'Renewable Share (Last 30 Days) %',
        'Projected Carbon Emissions (tCO2e)',
        'Estimated Carbon Savings vs Baseline (tCO2e)'
    ],
    'Value': [
        actual_recent_energy,
        forecast_energy_next_30,
        renewable_share_recent,
        carbon_projected_tonnes,
        carbon_savings_tonnes
    ]
})

kpi_table

## 5) Dashboard and Drill-Down Views

In [ ]:
history_plot = campus_daily[['date', 'energy_kwh']].copy()
history_plot['series'] = 'Actual Campus Energy'

forecast_plot = forecast_df[['date', 'ensemble_forecast_kwh']].copy()
forecast_plot = forecast_plot.rename(columns={'ensemble_forecast_kwh': 'energy_kwh'})
forecast_plot['series'] = 'Ensemble Forecast'

combined_plot = pd.concat([history_plot, forecast_plot], ignore_index=True)

fig_trend = px.line(
    combined_plot,
    x='date',
    y='energy_kwh',
    color='series',
    title='Campus Energy Trend with Ensemble Forecast',
    labels={'energy_kwh': 'Energy (kWh)', 'date': 'Date'}
)
fig_trend.update_layout(template='plotly_white')
fig_trend.show()

In [ ]:
zone_daily = (
    building_daily
    .groupby(['date', 'zone'], as_index=False)[['energy_kwh', 'water_kl', 'waste_kg', 'renewable_kwh']]
    .sum()
)

fig_zone_drill = px.line(
    zone_daily,
    x='date',
    y='energy_kwh',
    color='zone',
    title='Zone Drill-Down: Daily Energy Consumption',
    labels={'energy_kwh': 'Energy (kWh)', 'date': 'Date'}
)
fig_zone_drill.update_layout(template='plotly_white')
fig_zone_drill.show()

zone_kpi = (
    zone_daily.groupby('zone', as_index=False)
    .agg(total_energy_kwh=('energy_kwh', 'sum'), total_water_kl=('water_kl', 'sum'), total_waste_kg=('waste_kg', 'sum'))
)
zone_kpi

In [ ]:
building_latest_month = building_daily[building_daily['date'] >= (building_daily['date'].max() - pd.Timedelta(days=29))]
building_summary = (
    building_latest_month
    .groupby(['zone', 'building'], as_index=False)['energy_kwh']
    .sum()
    .sort_values(['zone', 'energy_kwh'], ascending=[True, False])
)

fig_building = px.bar(
    building_summary,
    x='building',
    y='energy_kwh',
    color='zone',
    title='Building Drill-Down: Last 30 Days Energy by Building',
    labels={'energy_kwh': 'Energy (kWh)', 'building': 'Building'}
)
fig_building.update_layout(template='plotly_white', xaxis_tickangle=-20)
fig_building.show()

building_summary.head(10)